# ES vs GA Comparison

Compare Vicuna ES and GA results using only the common query IDs. Optional convergence curves are derived from saved generation history CSVs.

## Summary Statistics

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style='whitegrid')
ROOT = Path('..').resolve()
MODEL_DIR = ROOT / 'outputs' / 'vicuna_7b'

def latest_run_with_algorithm(base: Path, algorithm_name: str) -> Path:
    runs = sorted(p for p in base.glob('run_*') if p.is_dir())
    for run_dir in reversed(runs):
        history_dir = run_dir / 'generation_history'
        if not history_dir.exists():
            continue
        history_files = sorted(history_dir.glob('*.csv'))
        if not history_files:
            continue
        sample = pd.read_csv(history_files[0])
        if 'algorithm' in sample.columns and str(sample['algorithm'].iloc[0]).strip().lower() == algorithm_name.lower():
            return run_dir
    raise FileNotFoundError(f'No {algorithm_name} run found under {base}')

def load_results(run_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(run_dir / 'final_results' / 'results.csv')
    df['query_id'] = df['query_id'].astype(str)
    return df

es_run = latest_run_with_algorithm(MODEL_DIR, 'es')
ga_run = latest_run_with_algorithm(MODEL_DIR, 'ga')
es = load_results(es_run)
ga = load_results(ga_run)

common = set(es['query_id']) & set(ga['query_id'])
print('Common queries:', len(common))
es = es[es['query_id'].isin(common)].copy()
ga = ga[ga['query_id'].isin(common)].copy()

es_success = es['attack_success'].astype(str).eq('True').mean()
ga_success = ga['attack_success'].astype(str).eq('True').mean()
print('ES success:', es_success)
print('GA success:', ga_success)
print('ES max:', pd.to_numeric(es['fitness']).max())
print('GA max:', pd.to_numeric(ga['fitness']).max())

## Visualization

In [ ]:
summary = pd.DataFrame({
    'method': ['ES', 'GA'],
    'success_rate': [es_success, ga_success],
    'best_fitness': [pd.to_numeric(es['fitness']).mean(), pd.to_numeric(ga['fitness']).mean()],
    'success_count': [es['attack_success'].astype(str).eq('True').sum(), ga['attack_success'].astype(str).eq('True').sum()],
})
display(summary)

plt.figure(figsize=(6, 4))
sns.barplot(data=summary, x='method', y='success_rate', palette='deep')
plt.title('Attack Success Rate')
plt.xlabel('Method')
plt.ylabel('Success Rate')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
plt.scatter(pd.to_numeric(es['fitness']), pd.to_numeric(ga['fitness']), alpha=0.7)
plt.xlabel('ES Fitness')
plt.ylabel('GA Fitness')
plt.title('ES vs GA Fitness')
plt.tight_layout()
plt.show()

diff = pd.to_numeric(ga['fitness']).reset_index(drop=True) - pd.to_numeric(es['fitness']).reset_index(drop=True)
plt.figure(figsize=(7, 4))
sns.histplot(diff, bins=20, kde=True, color='purple')
plt.title('GA minus ES Fitness Difference')
plt.xlabel('Fitness Difference')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
def load_history_curve(run_dir: Path) -> pd.DataFrame:
    history_dir = run_dir / 'generation_history'
    frames = []
    for csv_path in sorted(history_dir.glob('*.csv')):
        df = pd.read_csv(csv_path)
        if 'generation' in df.columns and 'best_fitness' in df.columns:
            frames.append(df[['generation', 'best_fitness']].copy())
    if not frames:
        return pd.DataFrame()
    combined = pd.concat(frames, ignore_index=True)
    return combined.groupby('generation', as_index=False)['best_fitness'].mean()

es_curve = load_history_curve(es_run)
ga_curve = load_history_curve(ga_run)
if not es_curve.empty and not ga_curve.empty:
    plt.figure(figsize=(8, 4))
    plt.plot(es_curve['generation'], es_curve['best_fitness'], marker='o', label='ES')
    plt.plot(ga_curve['generation'], ga_curve['best_fitness'], marker='o', label='GA')
    plt.xlabel('Generation')
    plt.ylabel('Mean Best Fitness')
    plt.title('Convergence Curves')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Convergence curves unavailable in saved history CSVs.')

## Insights

- Does crossover help? Inspect the scatter and difference plots to see whether GA consistently improves fitness over ES.
- Where does GA outperform ES? Positive values in the difference histogram indicate query cases where GA is stronger.
- If convergence curves are available, compare whether one optimizer reaches strong fitness faster.